# Single-Server FIFO Queue Simulation

This notebook presents a discrete-event simulation study of a single-server, FIFO, work-conserving queue with configurable arrival-process assumptions. The workflow compares exponential interarrivals with a matched-mean uniform alternative to examine how utilization and arrival variability affect congestion metrics.

## Imports and Reproducibility Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, time
np.random.seed(2025)
random.seed(2025)


## Core Functions

In [ ]:
def generate_poisson_arrivals(n_customers, lam):
    """
    Generate cumulative arrival times for a Poisson process with rate λ.
    Interarrival times are i.i.d. Exp(λ); returns a nondecreasing array of length n_customers.

    ----
    returns :np.ndarray
    """
    interarrival_times = np.random.exponential(scale=1.0 / lam, size=n_customers)
    cumulative_arrival_times = np.cumsum(interarrival_times)

    return cumulative_arrival_times


In [ ]:
def generate_uniform_arrivals(n_customers, lam):
    """
    Generate cumulative arrival times with i.i.d. uniform interarrival times.
    Useful for the 'wrong-model' comparison with reduced arrival variability.
    ----
    returns :np.ndarray
    """
    interarrival_times = np.random.uniform(low=0.0, high=2.0 / lam, size=n_customers)
    cumulative_arrival_times = np.cumsum(interarrival_times)

    return cumulative_arrival_times


In [ ]:
def generate_service_times(n_customers, mu):
    """
    Generate i.i.d. service durations S_i ~ Exp(μ) with mean 1/μ.

    ----
    returns :np.ndarray
    """
    service_times = np.random.exponential(scale=1.0 / mu, size=n_customers)

    return service_times


In [ ]:
def simulate_single_server_fifo(
    arrival_times,
    service_times,
):
    """
    Simulate a single-server, FIFO, work-conserving queue (M/G/1 structure with given service samples).

    Parameters
    ----------
    arrival_times : np.ndarray
        Nondecreasing arrival timestamps for each job (shape (N,)).
    service_times : np.ndarray
        Service durations for each job (shape (N,)).
    """

    n = len(arrival_times)
    start_times = np.zeros(n)
    departure_times = np.zeros(n)
    waiting_times = np.zeros(n)
    system_times = np.zeros(n)

    # This represents the first job. Since server is free, service starts on arrival
    start_times[0] = arrival_times[0]
    departure_times[0] = start_times[0] + service_times[0]

    for i in range(1, n):
        start_times[i] = max(departure_times[i - 1], arrival_times[i])
        departure_times[i] = start_times[i] + service_times[i]

    waiting_times = start_times - arrival_times
    system_times = departure_times - arrival_times

    return {
        "arrival_times": arrival_times,
        "service_times": service_times,
        "start_times": start_times,
        "departure_times": departure_times,
        "waiting_times": waiting_times,
        "system_times": system_times,
    }


In [ ]:
def compute_performance_metrics(sim_result):
    """
    Compute summary performance metrics from a completed simulation.

    Returns
    -------
    dict with W (mean wait), T (mean system time), rho (utilization),
    Lq (time-average queue length, via Little's Law: Lq = lambda_eff * W)
    """
    waiting_times = sim_result["waiting_times"]
    system_times = sim_result["system_times"]
    service_times = sim_result["service_times"]
    arrival_times = sim_result["arrival_times"]
    departure_times = sim_result["departure_times"]

    N = len(arrival_times)

    W = np.mean(waiting_times)
    T = np.mean(system_times)

    # Utilization: busy time of server / simulation horizon
    busy_time = np.sum(service_times)
    horizon = departure_times[-1] - arrival_times[0]
    rho = busy_time / horizon

    # Effective arrival rate over the simulation horizon
    lambda_eff = N / horizon

    Lq = lambda_eff * W

    return {"W": W, "T": T, "rho": rho, "Lq": Lq}


## Design Analysis (Exponential Interarrivals)

In [ ]:
def run_grid(pairs=((0.5,1.0),(0.7,1.0),(0.9,1.0),(0.95,1.0)), N=2000, arrivals_type='exp'):
    rho_e, W_e, T_e, Lq_e, U_e = [], [], [], [], []
    for lam, mu in pairs:
      if arrivals_type == 'exp':
          arrival_times = generate_poisson_arrivals(N, lam)
      elif arrivals_type == 'uniform':
          arrival_times = generate_uniform_arrivals(N, lam)
      else:
          raise ValueError("arrivals_type must be 'exp' or 'uniform'")

      service_times = generate_service_times(N, mu)
      sim_result = simulate_single_server_fifo(arrival_times, service_times)
      metrics = compute_performance_metrics(sim_result)

      rho_e.append(lam / mu)
      W_e.append(metrics["W"])
      T_e.append(metrics["T"])
      Lq_e.append(metrics["Lq"])
      U_e.append(metrics["rho"])

    return rho_e, W_e, T_e, Lq_e, U_e


In [ ]:
pairs = ((0.5, 1.0), (0.7, 1.0), (0.9, 1.0), (0.95, 1.0))
N = 2000

rho_e, W_e, T_e, Lq_e, U_e = run_grid(pairs=pairs, N=N, arrivals_type='exp')


## Figures 1–2

In [ ]:
# Figure 1: Wbar vs rho
plt.figure()
plt.plot(rho_e, W_e, marker='o')
plt.xlabel(r"Utilization $\rho = \lambda/\mu$")
plt.ylabel(r"Mean waiting time $\bar{W}$")
plt.title("Figure 1: Mean Waiting Time vs. Utilization (Exponential Interarrivals)")
plt.grid(True)
plt.savefig("figure1_W_vs_rho.pdf")
plt.show()


# Figure 2: Tbar vs rho
plt.figure()
plt.plot(rho_e, T_e, marker='o', color='darkorange')
plt.xlabel(r"Utilization $\rho = \lambda/\mu$")
plt.ylabel(r"Mean system time $\bar{T}$")
plt.title("Figure 2: Mean System Time vs. Utilization (Exponential Interarrivals)")
plt.grid(True)
plt.savefig("figure2_T_vs_rho.pdf")
plt.show()


## Example Numerical Summary

In [ ]:
lambdas = [lam for lam, mu in pairs]
mus     = [mu for lam, mu in pairs]

table1 = []
for lam, mu, rho, W, T, Lq, util in zip(lambdas, mus, rho_e, W_e, T_e, Lq_e, U_e):
    table1.append((lam, mu, rho, W, T, Lq, util))

header = f"{'lambda':>8} {'mu':>8} {'rho':>8} {'W':>10} {'T':>10} {'Lq':>10} {'utilization':>12}"
print(header)
print("-" * len(header))
for lam, mu, rho, W, T, Lq, util in table1:
    print(f"{lam:>8.2f} {mu:>8.2f} {rho:>8.2f} {W:>10.4f} {T:>10.4f} {Lq:>10.4f} {util:>12.4f}")


## Wrong-Model Comparison (Uniform Interarrivals)

In [ ]:
rho_u, W_u, T_u, Lq_u, U_u = run_grid(pairs=pairs, N=N, arrivals_type='uniform')


## Figures 3–4

In [ ]:
# Figure 3: Wbar vs rho (exp vs uniform)
plt.figure()
plt.plot(rho_e, W_e, marker='o', label='Exponential interarrivals')
plt.plot(rho_u, W_u, marker='s', label='Uniform interarrivals')
plt.xlabel(r"Utilization $\rho = \lambda/\mu$")
plt.ylabel(r"Mean waiting time $\bar{W}$")
plt.title("Figure 3: Mean Waiting Time vs. Utilization (Exp vs. Uniform Arrivals)")
plt.legend()
plt.grid(True)
plt.savefig("figure3_W_vs_rho_comparison.pdf")
plt.show()

# Figure 4: Lqbar vs rho (exp vs uniform)
plt.figure()
plt.plot(rho_e, Lq_e, marker='o', label='Exponential interarrivals')
plt.plot(rho_u, Lq_u, marker='s', label='Uniform interarrivals')
plt.xlabel(r"Utilization $\rho = \lambda/\mu$")
plt.ylabel(r"Mean queue length $\bar{L}_q$")
plt.title("Figure 4: Mean Queue Length vs. Utilization (Exp vs. Uniform Arrivals)")
plt.legend()
plt.grid(True)
plt.savefig("figure4_Lq_vs_rho_comparison.pdf")
plt.show()


## Discussion

Based on Figures 3 and 4 and the corresponding values in the summary table, reducing arrival variability yields a consistent reduction in both waiting time and queue length across the entire range of ρ. In both figures, the exponential-interarrival curve sits above the uniform-interarrival curve at every value of ρ, and the gap between the two widens noticeably as ρ increases toward 0.9–0.95. This shows that ρ alone does not determine congestion; the variability of the arrival process matters just as much, since both models share the same mean interarrival time 1/λ and the same μ, yet produce very different mean waiting time and mean queue length outcomes. The divergence is most severe at high utilization, exactly where an incorrect variability assumption would have the largest practical impact on predicted delay. In a real system-design context, relying on the uniform assumption when the true process is exponential would lead to under-provisioning server capacity or waiting space sized for the lower, 'wrong-model' queue lengths, only to see performance degrade sharply once the system operates near saturation. This illustrates that correctly modeling arrival variability, not just matching the mean arrival rate, is essential for making reliable capacity and design decisions as ρ approaches 1.